# Day 11 — Simplified: Why Words Need to Become Numbers

## The Question

A neural network only understands numbers. So how do you give it words?

```
"happy" → ??? → number(s) that the network can use
```

## The Bad Idea (Day 10)

Give each word a unique ID number, then make a vector that's all zeros except a 1 at that position. This is called **one-hot encoding**.

```
Vocab: ["happy", "sad", "amazing"]

"happy"   → [1, 0, 0]
"sad"     → [0, 1, 0]
"amazing" → [0, 0, 1]
```

**Two problems:**
1. With 50,000 words in a real vocabulary, each word is a vector of 49,999 zeros and one 1. Wasteful.
2. "happy" and "amazing" mean similar things — but their one-hot vectors are TOTALLY different. The model doesn't know they're related.

## The Good Idea (Day 11)

Give each word a small list of REAL numbers — a "fingerprint" that captures what the word means.

```
"happy"   → [0.8, 0.1, 0.9, -0.2]   ← positive vibe
"amazing" → [0.7, 0.2, 0.8, -0.1]   ← also positive! similar vector!
"sad"     → [-0.7, 0.3, -0.8, 0.1]  ← opposite vibe
```

**This is what we're going to build today.** These small dense vectors are called **embeddings**.

## Why does this matter?

When two words have similar meanings, their embeddings will be similar. The model can use this:
- It learns "great" is good → it automatically learns "amazing" is also good (similar vector)
- New word it never saw? If its embedding is close to "terrible", treat it like negative.

Embeddings are the FIRST thing every modern LLM does. GPT, Claude, Llama all start by looking up each word's embedding.

In [ ]:
import torch
import torch.nn as nn

# Embedding = a lookup table
# nn.Embedding(num_words, embedding_size)

# Imagine our vocab has 5 words
embedding = nn.Embedding(num_embeddings=5, embedding_dim=4)

# The table is just a (5 × 4) matrix of random numbers
print("The embedding table:")
print(embedding.weight.data)
print(f"\nShape: {embedding.weight.shape}  (5 words × 4 numbers each)")

In [ ]:
# How to use it: pass in word IDs, get back vectors

# Pretend word IDs 2, 0, 4 are some words
word_ids = torch.tensor([2, 0, 4])

# Embedding just looks up the corresponding rows
vectors = embedding(word_ids)
print(f"Word IDs {word_ids.tolist()} → vectors:")
print(vectors)
print(f"\nThat's it. Embedding = lookup row from the matrix.")

## What happens during training?

Right now the embedding values are random — they mean nothing. But the table is **learnable**.

During training:
1. The model sees "this movie was great" → label = positive
2. It uses the embeddings of "this", "movie", "was", "great"
3. If its prediction is wrong, gradients flow back and adjust those embeddings
4. Over time, "great" drifts toward a "positive" direction in the embedding space
5. "Amazing" — appearing in similar sentences — drifts the same way
6. Now they have similar vectors!

**The model isn't TOLD that "great" and "amazing" are similar.** It figures this out from how the words are used.

## Quick recap

```
Day 10:  word → one-hot (huge sparse vector, no meaning)
Day 11:  word → embedding (small dense vector, meaningful)
```

Embeddings are how every LLM turns text into something it can do math on.

**Tomorrow (Day 12):** A complete sentiment classifier project using everything we've learned.